# 🐍 Exemplo de rede neural

Exemplo prático do capítulo: 2.8  Exemplo de rede neural aplicada ao reconhecimento facial

In [1]:
!unzip /content/dataset.zip -d /content/

Archive:  /content/dataset.zip
   creating: /content/dataset/
   creating: /content/dataset/Woodrow_Stanley/
  inflating: /content/dataset/Woodrow_Stanley/Woodrow_Stanley_0001.jpg  
   creating: /content/dataset/Woody_Allen/
  inflating: /content/dataset/Woody_Allen/Woody_Allen_0001.jpg  
  inflating: /content/dataset/Woody_Allen/Woody_Allen_0002.jpg  
  inflating: /content/dataset/Woody_Allen/Woody_Allen_0003.jpg  
  inflating: /content/dataset/Woody_Allen/Woody_Allen_0004.jpg  
  inflating: /content/dataset/Woody_Allen/Woody_Allen_0005.jpg  
   creating: /content/dataset/Wu_Peng/
  inflating: /content/dataset/Wu_Peng/Wu_Peng_0001.jpg  
   creating: /content/dataset/Wu_Yi/
  inflating: /content/dataset/Wu_Yi/Wu_Yi_0001.jpg  
  inflating: /content/dataset/Wu_Yi/Wu_Yi_0002.jpg  
  inflating: /content/dataset/Wu_Yi/Wu_Yi_0003.jpg  
   creating: /content/dataset/Wycliffe_Grousbeck/
  inflating: /content/dataset/Wycliffe_Grousbeck/Wycliffe_Grousbeck_0001.jpg  
   creating: /content/dataset

In [56]:
# @title 1 - 2.8 Exemplo de rede neural aplicada ao reconhecimento facial

# ---------------------------------------------------------
# Importação das bibliotecas
# ---------------------------------------------------------
import os
import random
import numpy as np
import tensorflow as tf
from IPython.display import Image, display
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image

In [ ]:
# @title Configurações iniciais

IMG_SIZE = 160
EMBEDDING_SIZE = 200
MARGIN = 0.5
EPOCHS = 30

In [32]:
# @title  Função para carregar imagem

def carregar_imagem(img_path):
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img = image.img_to_array(img)
    img = img / 255.0
    return img


In [33]:
# @title Gerador de triplets

def triplet_generator(dataset_path, batch_size=16):
    pessoas = os.listdir(dataset_path)

    while True:
        anc, pos, neg = [], [], []

        for _ in range(batch_size):
            pessoa = random.choice(pessoas)
            imgs_pessoa = os.listdir(os.path.join(dataset_path, pessoa))

            if len(imgs_pessoa) < 2:
                continue

            a, p = random.sample(imgs_pessoa, 2)

            pessoa_neg = random.choice([x for x in pessoas if x != pessoa])
            n = random.choice(os.listdir(os.path.join(dataset_path, pessoa_neg)))

            anc.append(carregar_imagem(os.path.join(dataset_path, pessoa, a)))
            pos.append(carregar_imagem(os.path.join(dataset_path, pessoa, p)))
            neg.append(carregar_imagem(os.path.join(dataset_path, pessoa_neg, n)))

        yield (
            np.array(anc),
            np.array(pos),
            np.array(neg)
        ), np.zeros((batch_size,))

In [34]:
# @title Cria os embeddings

def criar_embedding_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    x = layers.Conv2D(32, (3, 3), activation="relu")(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, (3, 3), activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, (3, 3), activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)

    x = layers.Dense(EMBEDDING_SIZE)(x)
    outputs = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1))(x)

    return models.Model(inputs, outputs, name="EmbeddingModel")


embedding_model = criar_embedding_model()

In [35]:
# @title Modelo Triplet

input_a = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
input_p = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
input_n = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

emb_a = embedding_model(input_a)
emb_p = embedding_model(input_p)
emb_n = embedding_model(input_n)

triplet_model = models.Model(
    inputs=[input_a, input_p, input_n],
    outputs=[emb_a, emb_p, emb_n]
)

In [36]:
# @title Triplet Loss

def triplet_loss(_, y_pred):
    emb_dim = EMBEDDING_SIZE

    emb_a = y_pred[:, 0:emb_dim]
    emb_p = y_pred[:, emb_dim:2*emb_dim]
    emb_n = y_pred[:, 2*emb_dim:3*emb_dim]

    pos_dist = tf.reduce_sum(tf.square(emb_a - emb_p), axis=1)
    neg_dist = tf.reduce_sum(tf.square(emb_a - emb_n), axis=1)

    loss = tf.maximum(pos_dist - neg_dist + MARGIN, 0.0)
    return tf.reduce_mean(loss)


In [37]:
# @title Treinamento

dataset_path = "/content/dataset"

triplet_model.compile(
    optimizer="adam",
    loss=triplet_loss
)

In [39]:
# @title Teste do nosso modelo

def gerar_embedding(img_path):
    img = carregar_imagem(img_path)
    img = np.expand_dims(img, axis=0)
    return embedding_model.predict(img, verbose=0)


def distancia_euclidiana(e1, e2):
    return np.linalg.norm(e1 - e2)

img1 = "/content/dataset/Woody_Allen/Woody_Allen_0001.jpg"
img2 = "/content/dataset/Woody_Allen/Woody_Allen_0001.jpg"
img3 = "/content/dataset/Wu_Peng/Wu_Peng_0001.jpg"

e1 = gerar_embedding(img1)
e2 = gerar_embedding(img2)
e3 = gerar_embedding(img3)

print("Mesmo rosto:", distancia_euclidiana(e1, e2))
print("Rostos diferentes:", distancia_euclidiana(e1, e3))

Mesmo rosto: 0.0
Rostos diferentes: 0.39501667


In [55]:
# @title Teste2 Controle de acesso ao prédio

THRESHOLD = 0.3

def verificar_acesso(img_referencia, img_entrada):
    emb_ref = gerar_embedding(img_referencia)
    emb_ent = gerar_embedding(img_entrada)

    distancia = distancia_euclidiana(emb_ref, emb_ent)

    if distancia < THRESHOLD:
        display(Image(url="https://cdn.pixabay.com/photo/2021/08/09/23/26/verified-6534505_1280.png", width=200, height=200))
    else:
        display(Image(url="https://cdn.pixabay.com/photo/2015/06/09/16/12/no-access-803719_1280.png", width=200, height=200))

    return distancia

# Pessoa autorizada
img_autorizada = "/content/dataset/Woody_Allen/Woody_Allen_0001.jpg"

# Caso 1 – Mesma pessoa
img_entrada_1 = "/content/dataset/Woody_Allen/Woody_Allen_0001.jpg"

# Caso 2 – Pessoa diferente
img_entrada_2 = "/content/dataset/Wu_Peng/Wu_Peng_0001.jpg"


print("\n--- Senhor: Woody Allen pode entrar ---\n")
verificar_acesso(img_autorizada, img_entrada_1)

print("\n--- Senhor: Wu Peng Infelizmente não pode entrar ---\n")
verificar_acesso(img_autorizada, img_entrada_2)



--- Senhor: Woody Allen pode entrar ---




--- Senhor: Wu Peng Infelizmente não pode entrar ---



np.float32(0.39501667)